*<h3>Some strong evidence from the dataset:</h3>*

- Median listing price: $252.29

- 75th percentile: $438.56

- Maximum: $99,999

- 521 listings (8.8% of valid-price listings) are above the standard IQR

- upper-outlier boundary of $868.41

- 1,400 listings (19.1%) have missing prices

- Median price differs dramatically by room type:

    - Hotel room: $409.92

    - Entire home/apt: $298.00

    - Private room: $159.00

    - Shared room: $64.50

- Neighborhood differences are also substantial. For example, the median in North Beach is $520, while Ocean View is $141.97.

***

<h3>1. Problem Statement & Objectives</h1>

Airbnb listing prices vary substantially across neighborhoods and room types, while missing and extreme price values make simple price comparisons potentially misleading. This analysis aims to understand the main patterns behind listing-price variation and identify data-quality issues that may affect reliable price comparisons.





1. **Measure how listing prices vary across room types and neighborhoods** and identify the areas and accommodation types associated with higher or lower typical prices.

2. **Examine how listing characteristics, such as accommodates and minimum-night requirements, are associated with listing prices.**

3. **Identify missing and extreme price values and assess how they could affect price comparisons and EDA results.**

***


In [36]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("listings.csv")

print(df.shape)
df.head()

(7332, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,958,https://www.airbnb.com/rooms/958,20260614073227,2026-06-14,city scrape,"Bright, Modern Garden Unit - 1BR/1BTH",**Please note that the house behind our backya...,NaN,https://a0.muscache.com/pictures/be1bf5ac-a955...,1169,...,4.90,4.97,4.78,STR-0006854,NaN,1,1,0,0,2.58
1,5858,https://www.airbnb.com/rooms/5858,20260614073227,2026-06-14,city scrape,Creative Sanctuary,We live in a large Victorian house on a quiet ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,8904,...,4.85,4.77,4.68,NaN,NaN,1,1,0,0,0.50
2,8014,https://www.airbnb.com/rooms/8014,20260614073227,2026-06-14,city scrape,female HOST quiet fast internet market parking,Room is on the second floor so it gets a good ...,NaN,https://a0.muscache.com/pictures/2cc1fc3d-0ae0...,22402,...,4.95,4.60,4.67,STR-0000974,NaN,2,0,2,0,0.55
3,8142,https://www.airbnb.com/rooms/8142,20260614073227,2026-06-14,city scrape,*FriendlyRoom Apt. Style -UCSF/USF - San Franc...,Nice and good public transportation. 7 minute...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,21994,...,4.82,4.73,4.73,NaN,NaN,22,0,22,0,0.08
4,8339,https://www.airbnb.com/rooms/8339,20260614073227,2026-06-22,previous scrape,Historic Alamo Square Victorian,"For creative humans who love art, space, photo...",NaN,https://a0.muscache.com/pictures/miso/Hosting-...,24215,...,5.00,4.94,4.75,STR-0000264,NaN,1,1,0,0,0.13


In [37]:
df["price_num"] = (
    df["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["price_num"] = pd.to_numeric(df["price_num"], errors="coerce")

df["price_num"].describe()

count     5932.000000
mean       413.858972
std       1431.984941
min          4.140000
25%        152.000000
50%        252.290000
75%        438.562500
max      99999.000000
Name: price_num, dtype: float64

In [38]:
missing_price = df["price_num"].isna().sum()
missing_price_pct = df["price_num"].isna().mean() * 100

print("Missing price:", missing_price)
print("Missing percentage:", round(missing_price_pct, 2), "%")

Missing price: 1400
Missing percentage: 19.09 %


In [39]:
fig = px.histogram(
    df,
    x="price_num",
    nbins=80,
    title="Distribution of Airbnb Listing Prices",
    labels={"price_num": "Price ($)"}
)

fig.show()

In [40]:
fig = px.histogram(
    df[df["price_num"] <= 2000],
    x="price_num",
    nbins=60,
    title="Distribution of Airbnb Prices (Up to $2,000)",
    labels={"price_num": "Price ($)"}
)

fig.show()

In [41]:
fig = px.box(
    df,
    y="price_num",
    title="Airbnb Listing Price Box Plot",
    labels={"price_num": "Price ($)"}
)

fig.show()

In [42]:
price = df["price_num"].dropna()

Q1 = price.quantile(0.25)
Q3 = price.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["price_num"] < lower_bound) |
    (df["price_num"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(outliers))

Q1: 152.0
Q3: 438.5625
IQR: 286.5625
Upper bound: 868.40625
Number of outliers: 521


In [43]:
room_price = (
    df.groupby("room_type")["price_num"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)

room_price

,count,median,mean
room_type,,,
Hotel room,224,409.92,575.845223
Entire home/apt,3513,298.00,457.097284
Private room,2175,159.00,330.249577
Shared room,20,64.50,97.325000


In [44]:
fig = px.bar(
    room_price.reset_index(),
    x="room_type",
    y="median",
    title="Median Price by Room Type",
    labels={
        "room_type": "Room Type",
        "median": "Median Price ($)"
    }
)

fig.show()

In [45]:
neighborhood_price = (
    df.groupby("neighbourhood_cleansed")["price_num"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)

neighborhood_price.head(10)

,count,median,mean
neighbourhood_cleansed,,,
Presidio,23,540.000,544.592609
North Beach,241,520.000,672.904108
Seacliff,6,445.335,400.700000
Financial District,210,390.785,1206.082476
Russian Hill,174,362.715,559.435345
Presidio Heights,23,351.500,472.358696
Downtown/Civic Center,810,327.750,413.890679
Pacific Heights,128,292.910,711.188281
Chinatown,103,286.000,380.940680


In [46]:
top10 = neighborhood_price.head(10).reset_index()

fig = px.bar(
    top10.sort_values("median"),
    x="median",
    y="neighbourhood_cleansed",
    orientation="h",
    title="Top 10 Neighborhoods by Median Listing Price",
    labels={
        "median": "Median Price ($)",
        "neighbourhood_cleansed": "Neighborhood"
    }
)

fig.show()

In [47]:
bottom10 = neighborhood_price.tail(10).reset_index()

fig = px.bar(
    bottom10.sort_values("median"),
    x="median",
    y="neighbourhood_cleansed",
    orientation="h",
    title="10 Lowest-Priced Neighborhoods by Median Listing Price",
    labels={
        "median": "Median Price ($)",
        "neighbourhood_cleansed": "Neighborhood"
    }
)

fig.show()

In [48]:
accommodates_price = (
    df.groupby("accommodates")["price_num"]
    .agg(["count", "median", "mean"])
)

accommodates_price

,count,median,mean
accommodates,,,
1,575,84.100,131.239113
2,2373,185.320,307.760126
3,368,237.500,307.093777
4,1411,308.000,382.675500
5,231,411.000,517.849481
6,528,504.050,766.327311
7,59,545.710,818.763051
8,192,724.515,924.613177
9,23,581.400,1059.771304


In [49]:
fig = px.line(
    accommodates_price.reset_index(),
    x="accommodates",
    y="median",
    markers=True,
    title="Median Price by Number of Guests Accommodated",
    labels={
        "accommodates": "Accommodates",
        "median": "Median Price ($)"
    }
)

fig.show()

In [50]:
numeric_cols = [
    "price_num",
    "accommodates",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365"
]

corr = df[numeric_cols].corr()

corr

,price_num,accommodates,minimum_nights,number_of_reviews,reviews_per_month,availability_365
price_num,1.000000,0.164985,-0.070112,-0.043007,-0.042035,0.044273
accommodates,0.164985,1.000000,-0.093922,-0.065626,-0.003043,0.048310
minimum_nights,-0.070112,-0.093922,1.000000,-0.129193,-0.174665,0.096635
number_of_reviews,-0.043007,-0.065626,-0.129193,1.000000,0.654369,-0.014780
reviews_per_month,-0.042035,-0.003043,-0.174665,0.654369,1.000000,0.040157
availability_365,0.044273,0.048310,0.096635,-0.014780,0.040157,1.000000


In [51]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Correlation Between Numerical Variables"
)

fig.show()

In [52]:
bins = [0, 1, 3, 7, 14, 29, 90, np.inf]
labels = [
    "1 night",
    "2–3 nights",
    "4–7 nights",
    "8–14 nights",
    "15–29 nights",
    "30–90 nights",
    "90+ nights"
]

df["minimum_nights_group"] = pd.cut(
    df["minimum_nights"],
    bins=bins,
    labels=labels
)

In [53]:
min_nights_price = (
    df.groupby("minimum_nights_group", observed=True)["price_num"]
    .agg(["count", "median", "mean"])
)

min_nights_price

,count,median,mean
minimum_nights_group,,,
1 night,1949,329.800,574.652370
2–3 nights,1529,354.000,505.577560
4–7 nights,186,339.600,528.705806
8–14 nights,12,299.085,351.322500
15–29 nights,7,144.300,212.572857
30–90 nights,2179,159.130,208.765425
90+ nights,66,5.280,29.821515


In [54]:
fig = px.bar(
    min_nights_price.reset_index(),
    x="minimum_nights_group",
    y="median",
    title="Median Price by Minimum-Night Requirement",
    labels={
        "minimum_nights_group": "Minimum-Night Requirement",
        "median": "Median Price ($)"
    }
)

fig.show()

In [55]:
suspicious_prices = (
    df[df["price_num"] > upper_bound]
    .sort_values("price_num", ascending=False)
)

suspicious_prices[
    [
        "id",
        "price_num",
        "room_type",
        "property_type",
        "neighbourhood_cleansed",
        "accommodates",
        "minimum_nights"
    ]
].head(20)

,id,price_num,room_type,property_type,neighbourhood_cleansed,accommodates,minimum_nights
2060,35195461,99999.00,Private room,Room in boutique hotel,Financial District,2,1.0
1706,28047393,17308.00,Entire home/apt,Entire home,Castro/Upper Market,6,3.0
2113,35899798,17213.00,Entire home/apt,Entire townhouse,Castro/Upper Market,6,3.0
2121,36002747,10000.00,Private room,Room in boutique hotel,Downtown/Civic Center,2,1.0
2122,36002932,10000.00,Private room,Room in boutique hotel,Nob Hill,2,1.0
2135,36148137,10000.00,Private room,Room in hotel,South of Market,2,1.0
2136,36148199,10000.00,Private room,Room in hotel,South of Market,2,1.0
5558,1275621657441158881,9999.00,Private room,Room in hotel,Downtown/Civic Center,2,1.0
3261,54020696,7133.00,Entire home/apt,Entire home,Castro/Upper Market,7,1.0
2237,37909467,5950.00,Entire home/apt,Entire villa,Pacific Heights,6,3.0


In [56]:
percent_missing = df.columns[df.isna().all()]

print(list(percent_missing))

['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'instant_bookable']


In [57]:
print("Completely empty columns:", len(percent_missing))

Completely empty columns: 13


In [ ]:
from IPython.display import HTML, display

html = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">

<style>
    body {
        margin: 0;
        padding: 0;
        background: #f3f4f6;
        font-family: Arial, Helvetica, sans-serif;
    }

    .deck {
        width: 100%;
    }

    .slide {
        width: 1100px;
        height: 620px;
        margin: 30px auto;
        padding: 45px 55px;
        box-sizing: border-box;
        background: white;
        border-radius: 18px;
        box-shadow: 0 8px 30px rgba(0,0,0,0.10);
        position: relative;
        overflow: hidden;
    }

    .title-slide {
        display: flex;
        flex-direction: column;
        justify-content: center;
    }

    h1 {
        font-size: 42px;
        margin: 0 0 20px 0;
        color: #172033;
    }

    h2 {
        font-size: 31px;
        margin: 0 0 25px 0;
        color: #172033;
    }

    h3 {
        color: #374151;
        margin-bottom: 10px;
    }

    p {
        color: #4b5563;
        font-size: 20px;
        line-height: 1.55;
    }

    .subtitle {
        font-size: 23px;
        color: #6b7280;
        max-width: 800px;
    }

    .small {
        font-size: 15px;
        color: #6b7280;
    }

    .footer {
        position: absolute;
        bottom: 20px;
        right: 40px;
        color: #9ca3af;
        font-size: 14px;
    }

    .tag {
        display: inline-block;
        padding: 8px 14px;
        border-radius: 20px;
        background: #eef2ff;
        color: #3730a3;
        font-size: 14px;
        font-weight: bold;
        margin-bottom: 20px;
    }

    .problem-box {
        background: #f8fafc;
        border-left: 6px solid #4f46e5;
        padding: 22px 25px;
        border-radius: 10px;
        margin-bottom: 25px;
    }

    .problem-box p {
        margin: 0;
        font-size: 21px;
        color: #1f2937;
    }

    .objectives {
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 18px;
    }

    .objective {
        background: #f9fafb;
        border: 1px solid #e5e7eb;
        border-radius: 12px;
        padding: 20px;
    }

    .number {
        font-size: 25px;
        font-weight: bold;
        color: #4f46e5;
        margin-bottom: 10px;
    }

    .objective p {
        font-size: 17px;
        margin: 0;
    }

    .stats {
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 15px;
        margin-top: 25px;
    }

    .stat {
        background: #f9fafb;
        border-radius: 12px;
        padding: 20px;
        text-align: center;
        border: 1px solid #e5e7eb;
    }

    .stat-value {
        font-size: 29px;
        font-weight: bold;
        color: #111827;
    }

    .stat-label {
        margin-top: 8px;
        font-size: 14px;
        color: #6b7280;
    }

    .two-col {
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 25px;
    }

    .card {
        background: #f9fafb;
        padding: 22px;
        border-radius: 12px;
        border: 1px solid #e5e7eb;
    }

    table {
        width: 100%;
        border-collapse: collapse;
        margin-top: 15px;
        font-size: 16px;
    }

    th, td {
        padding: 11px;
        border-bottom: 1px solid #e5e7eb;
        text-align: left;
    }

    th {
        background: #f9fafb;
        color: #374151;
    }

    .insight {
        background: #eef2ff;
        border-radius: 10px;
        padding: 18px;
        margin-top: 18px;
        font-size: 18px;
        line-height: 1.5;
        color: #312e81;
    }

    .recommendations {
        display: grid;
        grid-template-columns: 1fr 1fr 1fr;
        gap: 18px;
    }

    .recommendation {
        padding: 20px;
        background: #f9fafb;
        border-radius: 12px;
        border: 1px solid #e5e7eb;
    }

    .recommendation h3 {
        font-size: 19px;
    }

    .recommendation p {
        font-size: 16px;
        margin: 0;
    }

    .section {
        margin-bottom: 22px;
    }

    .section-title {
        font-weight: bold;
        font-size: 18px;
        color: #374151;
        margin-bottom: 8px;
    }

</style>
</head>

<body>

<div class="deck">

    <!-- SLIDE 1 -->
    <section class="slide title-slide">

        <div class="tag">EXPLORATORY DATA ANALYSIS</div>

        <h1>Airbnb Listing Price Patterns<br>
        & Data Quality Analysis</h1>

        <p class="subtitle">
            Investigating price variation across listings and identifying
            data-quality issues that may affect reliable price comparisons.
        </p>

        <br>

        <p class="small">
            Data Science EDA Project<br>
            Your Name | General Assembly Data Science Bootcamp
        </p>

        <div class="footer">01 / 06</div>
    </section>


    <!-- SLIDE 2 -->
    <section class="slide">

        <h2>Problem Statement & Objectives</h2>

        <div class="problem-box">
            <p>
                Airbnb listing prices vary substantially across neighborhoods
                and room types, while missing and extreme price values make
                simple price comparisons potentially misleading.
            </p>
        </div>

        <h3>Objectives</h3>

        <div class="objectives">

            <div class="objective">
                <div class="number">01</div>
                <p>
                    Measure how listing prices vary across
                    room types and neighborhoods.
                </p>
            </div>

            <div class="objective">
                <div class="number">02</div>
                <p>
                    Examine how listing characteristics,
                    such as accommodates and minimum-night
                    requirements, are associated with price.
                </p>
            </div>

            <div class="objective">
                <div class="number">03</div>
                <p>
                    Identify missing and extreme price values
                    and assess how they may affect price analysis.
                </p>
            </div>

        </div>

        <div class="footer">02 / 06</div>
    </section>


    <!-- SLIDE 3 -->
    <section class="slide">

        <h2>Price Distribution & Data Quality</h2>

        <p>
            The price variable is highly skewed and contains a substantial
            amount of missing and extreme data.
        </p>

        <div class="stats">

            <div class="stat">
                <div class="stat-value">$252.29</div>
                <div class="stat-label">Median Price</div>
            </div>

            <div class="stat">
                <div class="stat-value">$99,999</div>
                <div class="stat-label">Maximum Price</div>
            </div>

            <div class="stat">
                <div class="stat-value">1,400</div>
                <div class="stat-label">Missing Prices</div>
            </div>

            <div class="stat">
                <div class="stat-value">521</div>
                <div class="stat-label">IQR Outliers</div>
            </div>

        </div>

        <div class="two-col" style="margin-top:25px;">

            <div class="card">
                <h3>Expected EDA Visual</h3>
                <p>
                    <b>Histogram:</b> Shows the strongly right-skewed
                    price distribution.
                </p>

                <p>
                    <b>Box Plot:</b> Highlights extreme observations.
                </p>
            </div>

            <div class="card">
                <h3>Key Insight</h3>
                <p>
                    The large difference between the median and maximum
                    price indicates that a small number of extreme listings
                    can strongly affect average-price analysis.
                </p>
            </div>

        </div>

        <div class="insight">
            <b>Data-quality concern:</b>
            19.1% of listings have missing price values, while 521
            valid-price observations fall above the IQR outlier threshold
            of approximately $868.
        </div>

        <div class="footer">03 / 06</div>
    </section>


    <!-- SLIDE 4 -->
    <section class="slide">

        <h2>Price Variation by Room Type & Neighborhood</h2>

        <div class="two-col">

            <div class="card">

                <h3>Median Price by Room Type</h3>

                <table>
                    <tr>
                        <th>Room Type</th>
                        <th>Median</th>
                    </tr>

                    <tr>
                        <td>Hotel room</td>
                        <td>$409.92</td>
                    </tr>

                    <tr>
                        <td>Entire home/apt</td>
                        <td>$298.00</td>
                    </tr>

                    <tr>
                        <td>Private room</td>
                        <td>$159.00</td>
                    </tr>

                    <tr>
                        <td>Shared room</td>
                        <td>$64.50</td>
                    </tr>
                </table>

            </div>

            <div class="card">

                <h3>Neighborhood Examples</h3>

                <table>
                    <tr>
                        <th>Neighborhood</th>
                        <th>Median</th>
                    </tr>

                    <tr>
                        <td>North Beach</td>
                        <td>$520.00</td>
                    </tr>

                    <tr>
                        <td>Financial District</td>
                        <td>$390.79</td>
                    </tr>

                    <tr>
                        <td>Russian Hill</td>
                        <td>$362.72</td>
                    </tr>

                    <tr>
                        <td>Ocean View</td>
                        <td>$141.97</td>
                    </tr>
                </table>

            </div>

        </div>

        <div class="insight">
            Listing prices differ substantially by accommodation type and location.
            For example, the median price in North Beach is more than 3.5× the median
            price in Ocean View.
        </div>

        <p class="small">
            Recommended visualizations: horizontal bar charts for room type
            and neighborhood median prices.
        </p>

        <div class="footer">04 / 06</div>
    </section>


    <!-- SLIDE 5 -->
    <section class="slide">

        <h2>Listing Characteristics & Price</h2>

        <div class="two-col">

            <div class="card">

                <h3>Accommodates vs. Median Price</h3>

                <table>
                    <tr>
                        <th>Guests</th>
                        <th>Median Price</th>
                    </tr>

                    <tr>
                        <td>1</td>
                        <td>$84.10</td>
                    </tr>

                    <tr>
                        <td>2</td>
                        <td>$185.32</td>
                    </tr>

                    <tr>
                        <td>4</td>
                        <td>$308.00</td>
                    </tr>

                    <tr>
                        <td>6</td>
                        <td>$504.05</td>
                    </tr>

                    <tr>
                        <td>8</td>
                        <td>$724.52</td>
                    </tr>
                </table>

            </div>

            <div class="card">

                <h3>Minimum-Night Requirement</h3>

                <p>
                    Median listing prices also vary according to the
                    minimum number of nights required for booking.
                </p>

                <p>
                    Short-stay and long-stay listings should therefore
                    be considered separately when comparing prices.
                </p>

                <div class="insight">
                    Larger-capacity listings generally have higher prices,
                    but capacity alone does not explain most of the
                    variation in listing prices.
                </div>

            </div>

        </div>

        <div class="insight">
            <b>Analytical takeaway:</b>
            Price should be evaluated using multiple listing characteristics
            rather than relying on a single overall average.
        </div>

        <p class="small">
            Recommended visualizations: line chart for accommodates vs.
            median price and grouped/bar chart for minimum-night categories.
        </p>

        <div class="footer">05 / 06</div>
    </section>


    <!-- SLIDE 6 -->
    <section class="slide">

        <h2>Recommendations, Limitations & Conclusion</h2>

        <h3>Recommendations</h3>

        <div class="recommendations">

            <div class="recommendation">
                <h3>01. Segment Price Benchmarks</h3>
                <p>
                    Compare prices by neighborhood and room type
                    instead of using one overall benchmark.
                </p>
            </div>

            <div class="recommendation">
                <h3>02. Review Extreme Values</h3>
                <p>
                    Investigate unusually high prices before using
                    the data for comparisons or modeling.
                </p>
            </div>

            <div class="recommendation">
                <h3>03. Handle Missing Data</h3>
                <p>
                    Establish a clear strategy for missing prices
                    before performing further analysis.
                </p>
            </div>

        </div>

        <div class="section" style="margin-top:22px;">
            <div class="section-title">Limitations</div>

            <p style="font-size:16px;">
                19.1% of listings have missing price values.
                Extreme prices cannot automatically be classified as errors.
                The dataset is observational, so correlations do not establish causation.
            </p>
        </div>

        <div class="section">
            <div class="section-title">Conclusion</div>

            <p style="font-size:17px;">
                Airbnb listing prices vary substantially across room types,
                neighborhoods, and listing characteristics. At the same time,
                missing and extreme price values create data-quality concerns.
                Reliable price comparison therefore requires segmentation
                and appropriate data-quality checks.
            </p>
        </div>

        <div class="footer">06 / 06</div>
    </section>

</div>

</body>
</html>
"""

display(HTML(html))